# Person Tracking


## 1. Configuration

In [ ]:
from pathlib import Path

# ── Project and input ──────────────────────────────────────────────────────
# The notebook works from either project/ or project/notebooks/.
WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
# Video to process. MP4/H.264 is the most reliable input for OpenCV.
INPUT_VIDEO = PROJECT_ROOT / "input" / "source04_fixed.mp4"
# First source timestamp to process, in seconds.
PROCESS_START_SECONDS = 0
# Final source timestamp; -1 processes through the end of the video.
PROCESS_END_SECONDS = 5

In [ ]:
mode = 'ANALYSIS'

## 2. Load the reusable project code

In [ ]:
import sys

%load_ext autoreload
%autoreload 2

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))


%load_ext person_tracker.notebook_magics

## 3. Check Python, FFmpeg, and the RTX GPU

In [ ]:
%%skip_if_mode PRODUCTION

import cv2
import torch
import ultralytics
from person_tracker.io import ffmpeg_available

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if CUDA_AVAILABLE else "cpu"
EFFECTIVE_TRACKING_QUANTIZE = 16 if CUDA_AVAILABLE else None
print("OpenCV:", cv2.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))
else:
    print("WARNING: YOLO tracking will run on CPU.")
print("OpenCV face APIs:", hasattr(cv2, "FaceDetectorYN") and hasattr(cv2, "FaceRecognizerSF"))
print("OpenCV CUDA devices:", cv2.cuda.getCudaEnabledDeviceCount() if hasattr(cv2, "cuda") else 0)
print("FFmpeg available:", ffmpeg_available())


## 4. Inspect the input and selected processing range

In [ ]:
%%skip_if_mode PRODUCTION

from person_tracker.io import inspect_video
from person_tracker.video import resolve_processing_range

if not INPUT_VIDEO.exists():
    raise FileNotFoundError(f"Place the source video at {INPUT_VIDEO}, or change INPUT_VIDEO in Section 1.")
video_info = inspect_video(INPUT_VIDEO)
processing_range = resolve_processing_range(video_info, PROCESS_START_SECONDS, PROCESS_END_SECONDS)
print(f"Resolution: {video_info.width} × {video_info.height}")
print(f"Frame rate: {video_info.fps:.3f} fps")
print(f"Duration: {video_info.duration_seconds:.2f} seconds")
print(
    f"Processing: {processing_range.start_seconds:.3f}s–{processing_range.end_seconds:.3f}s "
    f"({processing_range.frame_count:,} frames)"
)

## 5. Track Video

### Pass 1 -- Collect tracking + face evidence

In [ ]:
import contextlib
import cv2
import io
import warnings

from insightface.app import FaceAnalysis
from tqdm.auto import tqdm
from ultralytics import YOLO

from person_tracker.face import extract_track_faces

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="insightface.utils.face_align",
)

FACE_DETECTION_HZ = 15
MAX_FACES_PER_FRAME = 10

# 1. Load models
model = YOLO("yolo11m.pt")

with contextlib.redirect_stdout(io.StringIO()), \
     contextlib.redirect_stderr(io.StringIO()):
    face_app = FaceAnalysis(
        name="buffalo_l",
        providers=[
            "CUDAExecutionProvider",
            "CPUExecutionProvider",
        ],
    )

    face_app.prepare(
        ctx_id=0,
        det_size=(512, 512),
    )

# 2. Open input video
cap = cv2.VideoCapture(str(INPUT_VIDEO))

if not cap.isOpened():
    raise RuntimeError(f"Could not open {INPUT_VIDEO}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

face_detection_interval = max(1, round(fps / FACE_DETECTION_HZ))

start_frame = max(0, int(PROCESS_START_SECONDS * fps))
end_frame = (
    frame_count
    if PROCESS_END_SECONDS <= 0
    else min(
        frame_count,
        int(PROCESS_END_SECONDS * fps),
    )
)

cap.set(
    cv2.CAP_PROP_POS_FRAMES,
    start_frame,
)

tracking_history = {}
face_samples = []

# 3. Collect tracking and face observations
with tqdm(
    total=end_frame - start_frame,
    desc="Tracking + face analysis",
    unit="frame",
) as progress:
    for frame_no in range(start_frame, end_frame):
        ok, frame = cap.read()

        if not ok:
            break

        result = model.track(
            frame,
            persist=True,
            tracker="deepocsort.yaml",
            classes=[0],
            conf=0.25,
            device=0,
            verbose=False,
        )[0]

        frame_tracks = []

        if result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()

            for box, track_id, conf in zip(
                boxes,
                ids,
                confs,
            ):
                frame_tracks.append({
                    "track_id": int(track_id),
                    "bbox": box.astype(int),
                    "confidence": float(conf),
                })

            # Run face analysis at FACE_DETECTION_HZ.
            if (
                (frame_no - start_frame)
                % face_detection_interval
                == 0
            ):
                samples = extract_track_faces(
                    face_app,
                    frame,
                    frame_no,
                    boxes,
                    ids,
                    max_faces=MAX_FACES_PER_FRAME,
                )

                face_samples.extend(samples)

        tracking_history[frame_no] = frame_tracks
        progress.update(1)

cap.release()

print(f"Tracked frames: {len(tracking_history)}")
print(f"Face samples: {len(face_samples)}")

### Pass 2 -- Resolve logical Person IDs

In [ ]:
from person_tracker.identity import (
    build_identity_history_from_boundaries,
    build_track_identity_anchors,
    cluster_face_samples,
    detect_switch_boundaries,
    merge_face_clusters,
    resolve_frame_identity_conflicts,
    resolve_unidentified_tracks,
    smooth_identity_anchors,
    split_track_identity_anchors,
)

# 1. Cluster good-quality face embeddings into initial logical people.
valid_face_samples, face_assignments = cluster_face_samples(
    face_samples,
    min_quality=0.5,  # Ignore low-quality face crops
    eps=0.4,          # DBSCAN cosine-distance neighborhood threshold
    min_samples=4,    # Minimum nearby samples required to form a cluster
)

initial_person_count = len(set(face_assignments.values()))

# 2. Merge separate face clusters that strongly appear to be the same person.
face_assignments = merge_face_clusters(
    valid_face_samples,
    face_assignments,
    similarity_threshold=0.5,  # Minimum cosine similarity required to merge clusters
    top_k=5,                   # Best-quality faces used when comparing clusters
)

merged_person_count = len(set(face_assignments.values()))

# 3. Convert face-level Person IDs into per-track identity observations.
# Example: Track 12 -> [(frame 100, P3), (frame 110, P3), (frame 125, P4)]
track_identity_anchors = build_track_identity_anchors(
    valid_face_samples,
    face_assignments,
)

# 4. Smooth occasional noisy face assignments using nearby observations.
track_identity_anchors = {
    track_id: smooth_identity_anchors(
        anchors,
        window=5,  # Number of nearby face observations used for majority voting
    )
    for track_id, anchors in track_identity_anchors.items()
}

# 5. Split a tracker ID when face evidence suggests DeepOCSORT switched
# from one real person to another.
tracklets = split_track_identity_anchors(
    track_identity_anchors,
    min_segment_anchors=2,  # Minimum face observations required for an identity segment
)

# 6. Estimate the frame where each tracker-ID identity switch occurred.
switch_boundaries = detect_switch_boundaries(
    tracklets,
    tracking_history,
    min_jump_score=0.25,  # Minimum bbox discontinuity treated as a likely switch
)

# 7. Build initial frame-by-frame Person IDs from direct face/tracklet
# evidence and calculate identity confidence.
identity_history = build_identity_history_from_boundaries(
    tracking_history,
    tracklets,
    switch_boundaries,
    track_identity_anchors,
    frame_size=(width, height),  # Resolution of the analysis video
)

# 8. Resolve tracks that still have no Person ID.
# Search backward and forward in time for overlapping identified tracks.
identity_history = resolve_unidentified_tracks(
    tracking_history,
    identity_history,
    fps,
    max_search_seconds=1.0,       # Maximum search distance backward/forward
    min_overlap=0.15,             # Minimum IoU with an identified nearby track
    temporal_decay_seconds=1.0,   # How quickly distant evidence loses strength
    fallback_confidence_scale=0.75,  # Penalize inferred IDs compared with direct evidence
    min_fallback_confidence=0.10, # Leave very weak inferred identities unresolved
)

# 9. Enforce one-to-one Track ↔ Person assignments within each frame.
# Competing assignments are solved globally so one Person ID cannot be
# assigned to multiple active tracks at the same time.
identity_history = resolve_frame_identity_conflicts(
    tracking_history,
    identity_history,
    min_candidate_score=0.10,  # Ignore very weak Track ↔ Person candidates
)

print(f"Valid face samples: {len(valid_face_samples)}")
print(f"Persons before merge: {initial_person_count}")
print(f"Persons after merge: {merged_person_count}")
print(f"Tracks with face identity: {len(track_identity_anchors)}")

### Pass 3 -- Show Tracked Info

In [ ]:
%%skip_if_mode PRODUCTION

for track_id, segments in sorted(tracklets.items()):
    if len(segments) <= 1:
        continue

    print(f"\nTrack {track_id}")

    for i, segment in enumerate(segments):
        anchors = segment["anchors"]

        print(
            f"  Tracklet {i}: "
            f"P{segment['person_id']} | "
            f"{anchors[0][0]} → {anchors[-1][0]} | "
            f"{len(anchors)} faces"
        )

    for boundary in switch_boundaries.get(track_id, []):
        print(
            f"  Switch: "
            f"P{boundary['from_person']} → P{boundary['to_person']} | "
            f"{boundary['start_frame']}–{boundary['end_frame']} | "
            f"≈ {boundary['switch_frame']} | "
            f"score={boundary['score']:.2f}"
        )

In [ ]:
%%skip_if_mode PRODUCTION

from collections import Counter, defaultdict
import base64, cv2, pandas as pd

def img_html(sample):
    if sample is None:
        return ""

    _, buf = cv2.imencode(".jpg", sample.image)
    src = base64.b64encode(buf).decode()
    return f'<img src="data:image/jpeg;base64,{src}" width="70">'

samples_by_track_person = defaultdict(list)

for sample in valid_face_samples:
    person_id = face_assignments.get(id(sample))
    if person_id is not None:
        samples_by_track_person[(sample.track_id, person_id)].append(sample)

rows = []

for track_id, anchors in sorted(track_identity_anchors.items()):
    counts = Counter(person_id for _, person_id in anchors)
    person_ids = [person_id for person_id, _ in counts.most_common(3)]

    faces = []

    for person_id in person_ids:
        samples = samples_by_track_person[(track_id, person_id)]
        faces.append((
            person_id,
            max(samples, key=lambda s: s.quality),
        ))

    faces += [(None, None)] * (3 - len(faces))

    max_overlap = max(
        (
            identity.get("overlap", 0.0)
            for identities in identity_history.values()
            if (identity := identities.get(track_id))
        ),
        default=0.0,
    )

    rows.append({
        "Track": track_id,
        "Faces": len(anchors),
        "Persons": "<br/>".join(
            f"P{p} ({n})"
            for p, n in counts.most_common()
        ),
        "Face 1": (
            f"P{faces[0][0]}<br>{img_html(faces[0][1])}"
            if faces[0][0] is not None else ""
        ),
        "Face 2": (
            f"P{faces[1][0]}<br>{img_html(faces[1][1])}"
            if faces[1][0] is not None else ""
        ),
        "Face 3": (
            f"P{faces[2][0]}<br>{img_html(faces[2][1])}"
            if faces[2][0] is not None else ""
        ),
        "Max Overlap": f"{max_overlap:.2f}",
        "Frames": f"{anchors[0][0]} → {anchors[-1][0]}",
    })

display(pd.DataFrame(rows).style)

## 6. Display compiled output

### Show tracked faces info

In [ ]:
%%skip_if_mode PRODUCTION

import math
import cv2
import textwrap
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from collections import defaultdict

MAX_FACES_PER_PERSON = 50
FACES_PER_ROW = 10

# Group face samples by resolved Person ID.
faces_by_person = defaultdict(list)

for sample in valid_face_samples:
    person_id = face_assignments.get(id(sample))

    if person_id is not None:
        faces_by_person[person_id].append(sample)


def select_representative_faces(
    samples,
    max_faces=30,
):
    """
    Select representative faces across the person's full timeline.

    The timeline is divided into buckets and the highest-quality face
    from each bucket is selected. This avoids displaying many nearly
    identical faces from adjacent frames.

    Args:
        samples (list[FaceSample]):
            Face samples belonging to one resolved person.

        max_faces (int, optional):
            Maximum number of representative faces to return.
            Defaults to 30.

    Returns:
        list[FaceSample]:
            Representative samples ordered by frame number.
    """
    samples = sorted(
        samples,
        key=lambda sample: sample.frame_no,
    )

    if len(samples) <= max_faces:
        return samples

    selected = []

    for i in range(max_faces):
        start = round(
            i * len(samples) / max_faces
        )

        end = round(
            (i + 1) * len(samples) / max_faces
        )

        bucket = samples[start:end]

        if bucket:
            selected.append(
                max(
                    bucket,
                    key=lambda sample: sample.quality,
                )
            )

    return selected


for person_id in sorted(faces_by_person):
    all_samples = faces_by_person[person_id]

    samples = select_representative_faces(
        all_samples,
        max_faces=MAX_FACES_PER_PERSON,
    )

    track_ids = sorted({
        sample.track_id
        for sample in all_samples
    })

    rows = max(1, math.ceil(len(samples) / FACES_PER_ROW))

    fig, axes = plt.subplots(
        rows,
        FACES_PER_ROW,
        figsize=(
            FACES_PER_ROW * 2,
            rows * 2.4,
        ),
        squeeze=False,
    )

    display(HTML("<hr style='margin:30px 0 20px 0;'>"))
    fig.suptitle(
        f"Person {person_id} | {len(all_samples)} faces\n"
        f"{len(track_ids)} tracks: {textwrap.fill(str(track_ids), width=150)}",
        fontsize=14,
        y=0.95,
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    previous_track_id = None

    for i, sample in enumerate(samples):
        row = i // FACES_PER_ROW
        col = i % FACES_PER_ROW
        ax = axes[row][col]

        image = cv2.cvtColor(
            sample.image,
            cv2.COLOR_BGR2RGB,
        )

        ax.imshow(image)
        ax.axis("off")

        track_changed = (
            previous_track_id is not None
            and sample.track_id != previous_track_id
        )

        title = (
            f"T:{sample.track_id} "
            f"F:{sample.frame_no}\n"
            f"D:{sample.confidence * 100:.0f} "
            f"Q:{sample.quality:.2f}"
        )

        if track_changed:
            title += "\nTRACK CHANGE"

        ax.set_title(
            title,
            fontsize=7,
        )

        previous_track_id = sample.track_id

    # Hide unused cells.
    for i in range(
        len(samples),
        rows * FACES_PER_ROW,
    ):
        row = i // FACES_PER_ROW
        col = i % FACES_PER_ROW

        axes[row][col].axis("off")

    plt.tight_layout(
        rect=[0, 0, 1, 0.95]
    )

    plt.show()

### Render Phase 1 Video

In [ ]:
%%skip_if_mode PRODUCTION

import cv2

from tqdm.auto import tqdm
from person_tracker.drawing import (
    get_last_face_confidence,
    put_text,
    put_tracking_label,
    build_face_samples_by_track
)
from person_tracker.video import render_video

phase1_tracked_video = PROJECT_ROOT / "output" / "phase1_tracked.mp4"

face_samples_by_track = build_face_samples_by_track(valid_face_samples)

LEGEND_TEXT = "Track | Person | Identity | Face"
LEGEND_FONT_SCALE = 0.9
LEGEND_THICKNESS = 2
LEGEND_MARGIN = 20
LEGEND_WIDTH = cv2.getTextSize(
    LEGEND_TEXT,
    cv2.FONT_HERSHEY_SIMPLEX,
    LEGEND_FONT_SCALE,
    LEGEND_THICKNESS,
)[0][0]


def draw_tracking_legend(frame):
    x = max(0, frame.shape[1] - LEGEND_WIDTH - LEGEND_MARGIN - 10)

    put_text(
        frame,
        LEGEND_TEXT,
        x,
        40,
        font_scale=LEGEND_FONT_SCALE,
        thickness=LEGEND_THICKNESS,
    )


def draw_tracking_frame(frame_no, frame):
    tracks = tracking_history.get(frame_no, [])
    identities = identity_history.get(frame_no, {})
    identified = 0

    draw_tracking_legend(frame)

    for track in tracks:
        track_id = track["track_id"]
        x1, y1, x2, y2 = track["bbox"]
        identity = identities.get(track_id)

        if identity:
            identified += 1
            person_id = identity["person_id"]
            identity_confidence = round(identity["confidence"] * 100)

            face_confidence, _ = get_last_face_confidence(
                face_samples_by_track,
                track_id,
                frame_no,
            )

            face_confidence = (
                "--"
                if face_confidence is None
                else round(face_confidence * 100)
            )

            values = [
                track_id,
                person_id,
                identity_confidence,
                face_confidence,
            ]

            color = (0, 255, 0)
        else:
            values = [track_id, "--", "--", "--"]
            color = (0, 0, 255)

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        put_tracking_label(frame, x1, y1 - 8, values)

    total = len(tracks)

    put_text(frame, f"Frame: {frame_no}", 20, 40)
    put_text(frame, f"Tracks: {total}", 20, 75)
    put_text(frame, f"Identified: {identified}", 20, 110)
    put_text(frame, f"Unknown: {total - identified}", 20, 145)

    return frame

progress = tqdm(
    total=end_frame - start_frame,
    desc="Rendering",
    unit="frame",
)

def on_progress(done, total):
    progress.n = done
    progress.refresh()


result = render_video(
    INPUT_VIDEO,
    phase1_tracked_video,
    draw_frame=draw_tracking_frame,
    start_frame=start_frame,
    end_frame=end_frame,
    progress_callback=on_progress,
)

progress.close()

print(f"Saved: {result['path']}")
print(f"Frames: {result['frames']}")
print(f"Duration: {result['duration_seconds']:.2f}s")

### Display compiled video

In [ ]:
%%skip_if_mode PRODUCTION

from IPython.display import Video, display
from person_tracker.io import create_browser_preview

final_path = phase1_tracked_video
browser_preview_video = PROJECT_ROOT / "output" / "preview.mp4"
browser_preview_width = 980
final_path = Path(final_path).resolve()
browser_preview_path = create_browser_preview(
    final_path, browser_preview_video, width=browser_preview_width
)
display(Video(
    filename=str(browser_preview_path), embed=True,
    width=browser_preview_width,
    html_attributes="controls playsinline preload='metadata'",
))

## Targeted Person Tracking

In [ ]:
from collections import defaultdict

# Build galleries
person_galleries = defaultdict(list)

for sample in valid_face_samples:
    person_id = face_assignments.get(id(sample))
    if person_id is not None:
        person_galleries[person_id].append(sample)

person_galleries = dict(person_galleries)

# Build stats from final identity history
person_stats = defaultdict(lambda: {
    "frames": 0,
    "tracks": set(),
})

for identities in identity_history.values():
    for track_id, identity in identities.items():
        person_id = identity["person_id"]

        person_stats[person_id]["frames"] += 1
        person_stats[person_id]["tracks"].add(track_id)

person_stats = {
    person_id: {
        "frames": stats["frames"],
        "duration": stats["frames"] / fps,
        "tracks": stats["tracks"],
        "track_count": len(stats["tracks"]),
    }
    for person_id, stats in person_stats.items()
}

# Only show people that survived final identity resolution
person_galleries = {
    person_id: samples
    for person_id, samples in person_galleries.items()
    if person_id in person_stats
}

In [ ]:
from person_tracker.ui import select_person_widget

target_person_id = await select_person_widget(
    person_galleries,
    person_stats,
)

In [ ]:
print(f'Selected person : {target_person_id}')

In [ ]:
from person_tracker.framing import (
    SmoothContainmentBox,
    build_lookahead_boxes,
    build_reappearance_transitions,
    build_smooth_boxes,
    build_target_frames
)
from person_tracker.drawing import (
    build_face_samples_by_track,
)


HORIZONTAL_PADDING = 0.0
VERTICAL_PADDING = 0.05

face_samples_by_track = build_face_samples_by_track(valid_face_samples)

target_frames = build_target_frames(
    tracking_history,
    identity_history,
    target_person_id,
    start_frame,
    end_frame,
    aspect_ratio=(9, 16),
    horizontal_padding=HORIZONTAL_PADDING,
    vertical_padding=VERTICAL_PADDING,
)

reappearance_transitions = build_reappearance_transitions(
    target_frames,
    start_frame,
    end_frame,
)

lookahead_boxes = build_lookahead_boxes(
    target_frames,
    start_frame,
    end_frame,
    fps,
    aspect_ratio=(9, 16),
    lookahead_seconds=0.75,  # How far ahead framing planning can see
)

smooth_box = SmoothContainmentBox(
    fps=fps,
    aspect_ratio=(9, 16),
    response_time=0.35,          # Overall camera smoothness/responsiveness
    visible_expansion=0.10,      # Breathing room around visible person
    transition_expansion=0.20,   # Extra room during disappearance/reappearance
    containment_priority=0.50,   # Smoothness ↔ containment tradeoff
    max_position_speed=1.25,     # Max center movement per second
    max_zoom_speed=0.60,         # Max zoom change per second
)

smooth_boxes = build_smooth_boxes(
    target_frames,
    reappearance_transitions,
    start_frame,
    end_frame,
    smooth_box,
    lookahead_boxes=lookahead_boxes,
)

In [ ]:
%%skip_if_mode PRODUCTION

import cv2

from tqdm.auto import tqdm
from person_tracker.drawing import (
    build_face_samples_by_track,
    get_last_face_confidence,
    put_text,
    put_tracking_label,
)
from person_tracker.video import render_video

phase2_tracked_video = PROJECT_ROOT / "output" / "phase2_target.mp4"

def draw_tracking_frame(frame_no, frame):
    target = target_frames.get(frame_no)
    smooth_bbox = smooth_boxes.get(frame_no)

    if target:
        track = target["track"]
        identity = target["identity"]
        container_box = target["container_box"]

        track_id = track["track_id"]
        x1, y1, x2, y2 = track["bbox"]

        identity_confidence = round(identity["confidence"] * 100)

        face_confidence, _ = get_last_face_confidence(
            face_samples_by_track,
            track_id,
            frame_no,
        )
        face_confidence = "--" if face_confidence is None else round(face_confidence * 100)

        # Actual tracking box.
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Required container box with padding.
        ox1, oy1, ox2, oy2 = container_box
        cv2.rectangle(frame, (ox1, oy1), (ox2, oy2), (255, 0, 255), 3)

        put_tracking_label(
            frame,
            x1,
            y1 - 8,
            [track_id, target_person_id, identity_confidence, face_confidence],
        )

    # Final smooth framing/crop box.
    if smooth_bbox:
        sx1, sy1, sx2, sy2 = smooth_bbox
        cv2.rectangle(frame, (sx1, sy1), (sx2, sy2), (255, 255, 0), 3)

    status = "Visible" if target else "Target not visible"
    put_text(frame, f"Frame: {frame_no} | {status}", 20, 40)

    return frame


progress = tqdm(
    total=end_frame - start_frame,
    desc=f"Rendering Person {target_person_id}",
    unit="frame",
)

def on_progress(done, total):
    progress.n = done
    progress.refresh()

result = render_video(
    INPUT_VIDEO,
    phase2_tracked_video,
    draw_frame=draw_tracking_frame,
    start_frame=start_frame,
    end_frame=end_frame,
    progress_callback=on_progress,
)

progress.close()

print(f"Saved: {result['path']}")
print(f"Frames: {result['frames']}")
print(f"Duration: {result['duration_seconds']:.2f}s")

In [ ]:
%%skip_if_mode PRODUCTION

from IPython.display import Video, display
from person_tracker.io import create_browser_preview

final_path = phase2_tracked_video
browser_preview_video = PROJECT_ROOT / "output" / "preview.mp4"
browser_preview_width = 1500
final_path = Path(final_path).resolve()
browser_preview_path = create_browser_preview(
    final_path, browser_preview_video, width=browser_preview_width
)
display(Video(
    filename=str(browser_preview_path), embed=True,
    width=browser_preview_width,
    html_attributes="controls playsinline preload='metadata'",
))

## Final Video

In [ ]:
from tqdm.auto import tqdm
from person_tracker.framing import crop_frame
from person_tracker.video import render_video

final_video = PROJECT_ROOT / "output" / "final_cropped.mp4"

OUTPUT_SIZE = (1080, 1920)


def crop_tracking_frame(frame_no, frame):
    bbox = smooth_boxes.get(frame_no)

    if bbox is None:
        return cv2.resize(frame, OUTPUT_SIZE, interpolation=cv2.INTER_AREA)

    return crop_frame(frame, bbox, OUTPUT_SIZE)


progress = tqdm(
    total=end_frame - start_frame,
    desc=f"Cropping Person {target_person_id}",
    unit="frame",
)

def on_progress(done, total):
    progress.n = done
    progress.refresh()

result = render_video(
    INPUT_VIDEO,
    final_video,
    draw_frame=crop_tracking_frame,
    start_frame=start_frame,
    end_frame=end_frame,
    output_size=OUTPUT_SIZE,
    preset="p6",
    quality=18,
    progress_callback=on_progress,
)

progress.close()

print(f"Saved: {result['path']}")
print(f"Frames: {result['frames']}")
print(f"Duration: {result['duration_seconds']:.2f}s")

In [ ]:
from IPython.display import Video, display

display(Video(str(final_video), embed=True, width=540))